# 사출성형기 AI 데이터셋 — 준지도 학습 (가이드북 기반)

KAMP 「사출성형기 AI 데이터셋」 분석실습 가이드북 **2.3.2 준지도 학습 활용 알고리즘**([코드 27]~[코드 54])을 바탕으로 만든 노트북입니다.

**흐름**
1. 설정 · 라이브러리
2. 데이터 확인 (양품/불량 개수, LH/RH 샷 구조)
3. DataLoader — 학습/평가 분할 ([코드 48])
4. 유틸 함수 — 확신도, 평가지표 ([코드 46], [코드 47], [코드 52])
5. (선택) 하이퍼파라미터 탐색 ([코드 49])
6. sklearn 준지도 학습 — SVM / Random Forest / Gaussian NB ([코드 50])
7. Keras 심층 신경망 준지도 학습 ([코드 53], [코드 54])
8. 결과 비교 ([표 4] 형식)

**가이드북 코드에서 바꾼 부분**

| 항목 | 가이드북 | 이 노트북 |
|---|---|---|
| pseudo-label 합치기 | `pd.DataFrame(model.predict(...))`로 만들어서 컬럼 이름이 `0` → 기존 라벨(`PassOrFail`)과 안 맞음 | `PassOrFail` 이름의 Series로 만들고 index 재설정 |
| 모델 | SVM 코드만 전체 공개 | 같은 루프를 SVM / RF / GNB에 공통 적용 (`make_model` 인자) |
| RF 파라미터 | `max_features='auto'` | 최신 sklearn에서 제거됨 → `'sqrt'` (분류에서 같은 의미) |
| 신경망 저장 | `.h5` | `.keras` (최신 Keras 권장 형식) |
| 비라벨 데이터 양 | 전체 사용 | `UNLABELED_SAMPLE`로 샘플링 가능 (SVM은 수만 행에서 매우 느림). `None`이면 전체 사용 |
| 분할 방식 | 행 단위 층화 분할 | `SPLIT_MODE='guidebook'`이면 동일, `'shot'`이면 같은 샷(LH/RH)을 같은 쪽에 묶어서 분할 |
| 평가지표 | Accuracy, Precision, Recall, ROC-AUC(0/1 예측값 기준), F1, 혼동행렬 | 동일 + 확률 기반 PR-AUC 추가 |

## 1. 설정

In [ ]:
# ===== 사용자 설정 =====
DATA_DIR = './data'          # moldset_*.csv 4개가 있는 폴더 경로
PRODUCT = 'cn7'              # 'cn7' 또는 'rg3'
SPLIT_MODE = 'guidebook'     # 'guidebook': 가이드북과 동일(행 단위) / 'shot': LH·RH 같은 샷을 같은 쪽으로
TEST_SIZE = 0.3              # 가이드북 [코드 48]
SEED = 42

PERCENTAGE = 10              # 매 반복마다 pseudo-label을 붙일 비율(%) — 가이드북 기본값
UNLABELED_USAGE = 90         # 비라벨 데이터 중 몇 %를 쓸지 — 가이드북 기본값
UNLABELED_SAMPLE = 5000      # 비라벨 데이터 샘플 수. None이면 전체(3.5만 행, SVM은 수 시간 걸릴 수 있음)

RUN_GRID_SEARCH = False      # [코드 49] 하이퍼파라미터 탐색 실행 여부 (오래 걸림)
RUN_SKLEARN = ['SVM', 'RF', 'GNB']
RUN_DNN = True
DNN_EPOCHS_PER_ITER = 100  # 가이드북 값. 반복(약 22회)마다 최대 100 epoch → CPU에서 15~30분 걸릴 수 있음

## 2. 라이브러리 ([코드 27])

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)

warnings.filterwarnings('ignore')
np.random.seed(SEED)
pd.set_option('display.max_columns', 30)

LABELED_PATH = os.path.join(DATA_DIR, f'moldset_labeled_{PRODUCT}.csv')
UNLABELED_PATH = os.path.join(DATA_DIR, f'moldset_unlabeled_{PRODUCT}.csv')
print(LABELED_PATH, os.path.exists(LABELED_PATH))
print(UNLABELED_PATH, os.path.exists(UNLABELED_PATH))

## 3. 데이터 확인 ([코드 28]~[코드 42])

제공 파일은 가이드북의 전처리가 이미 끝난 상태입니다.
- `PassOrFail`: 0 = 양품, 1 = 불량 ([코드 41] 변환 완료)
- 공정변수 24개: **파일마다 따로** StandardScaler 적용 ([코드 43]) → labeled와 unlabeled의 스케일 기준이 다름

In [ ]:
labeled_raw = pd.read_csv(LABELED_PATH, low_memory=False)
unlabeled_raw = pd.read_csv(UNLABELED_PATH, low_memory=False)

FEATURES = [c for c in labeled_raw.columns if c not in ('Unnamed: 0', 'PassOrFail')]
print('labeled  :', labeled_raw.shape, '| unlabeled:', unlabeled_raw.shape, '| 변수 수:', len(FEATURES))

# [코드 42] 양품 / 불량 개수
print('양품 수:', (labeled_raw['PassOrFail'] == 0).sum())
print('불량 수:', (labeled_raw['PassOrFail'] == 1).sum())

### 3-1. LH/RH 샷 구조 확인

CN7·RG3는 한 샷에 LH·RH가 같이 찍혀서, **공정변수가 완전히 같은 행이 두 줄씩** 들어 있습니다.
같은 공정값을 가진 행끼리 `shot_id`를 붙이고, 한쪽만 불량인 샷이 몇 개인지 확인합니다.

In [ ]:
labeled_raw['shot_id'] = labeled_raw.groupby(FEATURES, sort=False).ngroup()

shot = labeled_raw.groupby('shot_id')['PassOrFail'].agg(['size', 'sum'])
print('행 수:', len(labeled_raw), '| 샷 수:', len(shot))
print('샷당 행 수 분포:', shot['size'].value_counts().to_dict())
print('한쪽만 불량인 샷 :', ((shot['sum'] > 0) & (shot['sum'] < shot['size'])).sum())
print('양쪽 다 불량인 샷:', ((shot['sum'] == shot['size']) & (shot['sum'] > 0)).sum())

## 4. DataLoader — 학습/평가 분할 ([코드 48])

- `SPLIT_MODE='guidebook'`: 가이드북과 같이 **행 단위** StratifiedShuffleSplit (test 30%)
- `SPLIT_MODE='shot'`: **샷 단위**로 층화 분할한 뒤 행으로 펼침 → 같은 샷의 LH·RH가 train/test에 나뉘지 않음

분할 후 **test 행 중 train에 똑같은 공정값이 있는 행**의 수를 출력합니다 (누수 확인).

In [ ]:
class DataLoader():
    def __init__(self, split_mode=SPLIT_MODE, test_size=TEST_SIZE, seed=SEED,
                 unlabeled_sample=UNLABELED_SAMPLE):
        # 클래스 변수가 존재하는 데이터
        labeled = labeled_raw.copy()
        # 클래스 변수가 존재하지 않는 데이터
        unlabeled = unlabeled_raw.drop(columns=['Unnamed: 0'])
        if unlabeled_sample is not None and unlabeled_sample < len(unlabeled):
            unlabeled = unlabeled.sample(unlabeled_sample, random_state=seed)
        unlabeled = unlabeled.reset_index(drop=True)

        sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        if split_mode == 'guidebook':
            train_idx, test_idx = next(sss.split(labeled[FEATURES], labeled['PassOrFail']))
        elif split_mode == 'shot':
            shot_y = labeled.groupby('shot_id')['PassOrFail'].max()   # 샷 안에 불량이 하나라도 있으면 1
            tr_s, te_s = next(sss.split(shot_y.index.to_frame(), shot_y.values))
            tr_ids, te_ids = shot_y.index[tr_s], shot_y.index[te_s]
            train_idx = np.where(labeled['shot_id'].isin(tr_ids))[0]
            test_idx = np.where(labeled['shot_id'].isin(te_ids))[0]
        else:
            raise ValueError(split_mode)

        tr, te = labeled.iloc[train_idx], labeled.iloc[test_idx]
        # 학습 / 평가 데이터
        self.moldset_labeled_train_X = tr[FEATURES].reset_index(drop=True)
        self.moldset_labeled_train_Y = tr['PassOrFail'].reset_index(drop=True)
        self.moldset_labeled_test_X = te[FEATURES].reset_index(drop=True)
        self.moldset_labeled_test_Y = te['PassOrFail'].reset_index(drop=True)
        # 클래스 변수가 없는 데이터
        self.moldset_unlabeled = unlabeled[FEATURES]

        # 누수 확인: test 행과 공정값이 똑같은 행이 train에 있는가
        self.leak_rows = int(te['shot_id'].isin(tr['shot_id']).sum())

data = DataLoader()
print('split_mode:', SPLIT_MODE)
print('train:', data.moldset_labeled_train_X.shape, '불량', int(data.moldset_labeled_train_Y.sum()))
print('test :', data.moldset_labeled_test_X.shape, '불량', int(data.moldset_labeled_test_Y.sum()))
print('unlabeled:', data.moldset_unlabeled.shape)
print('test 중 train에 같은 공정값이 있는 행:', data.leak_rows)

## 5. 유틸 함수 ([코드 46], [코드 47], [코드 52])

In [ ]:
# [코드 46] 확신도: 클래스별 확률 중 큰 값 (sklearn 모델용)
def confident_prediction(prob):
    result = []
    for i in range(len(prob)):
        if prob[i][0] >= prob[i][1]:
            result.append(prob[i][0])
        else:
            result.append(prob[i][1])
    return result

# [코드 46] 확신도: 신경망용 (출력이 불량 확률 하나)
def confident_prediction_1(prob):
    result = []
    for i in range(len(prob)):
        p = float(np.ravel(prob[i])[0])
        result.append(p if p >= 0.5 else 1 - p)
    return result

# [코드 52] 신경망 출력(확률) → 클래스(0/1)
def predict(prob):
    return [1 if float(np.ravel(p)[0]) >= 0.5 else 0 for p in prob]

# [코드 47] 평가지표 (가이드북과 동일한 출력)
def evaluation(y, y_pred):
    print('Accuracy: {:.2f}'.format(accuracy_score(y, y_pred)))
    print('Precision: {:.2f}'.format(precision_score(y, y_pred, zero_division=0)))
    print('Recall: {:.2f}'.format(recall_score(y, y_pred)))
    print(roc_auc_score(y, y_pred))
    print(f1_score(y, y_pred))
    print(confusion_matrix(y, y_pred))

# 결과 표용 (가이드북 지표 + 확률 기반 PR-AUC)
def score_dict(y, y_pred, y_score=None):
    d = {
        '정확도': accuracy_score(y, y_pred),
        '정밀도': precision_score(y, y_pred, zero_division=0),
        '재현율': recall_score(y, y_pred),
        'ROC-AUC(0/1 기준)': roc_auc_score(y, y_pred),
        'F1': f1_score(y, y_pred),
    }
    d['PR-AUC(확률 기준)'] = average_precision_score(y, y_score) if y_score is not None else np.nan
    tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0, 1]).ravel()
    d.update({'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp})
    return d

## 6. (선택) 하이퍼파라미터 탐색 ([코드 49])

`RUN_GRID_SEARCH = True`일 때만 실행합니다. 가이드북 탐색 범위는 C·gamma 각 99개(총 9,801조합 × 5-fold × 지표 3개)라 매우 오래 걸려서, 기본값은 범위를 줄여 두었습니다.

가이드북에 적힌 결과:
- SVM: `{'C': 0.197, 'gamma': 0.71, 'kernel': 'rbf'}` (단, [코드 50]의 SVM은 `C=0.001, gamma=1e-2`를 사용)
- RF: `{'bootstrap': False, 'max_depth': 10, 'max_features': 'auto', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 1200}`

In [ ]:
if RUN_GRID_SEARCH:
    tuned_parameters = [{'kernel': ['rbf'],
                         'gamma': [i * 1e-2 for i in range(1, 100, 10)],   # 가이드북: i*1e-4, i in 1..99
                         'C': [i * 1e-2 for i in range(1, 100, 10)]}]
    scores = ['accuracy', 'precision', 'recall']
    weights = {0: 100.0, 1: 1.0}
    kf = StratifiedKFold(random_state=SEED, n_splits=5, shuffle=True)
    X, y = data.moldset_labeled_train_X, data.moldset_labeled_train_Y
    for score in scores:
        clf = GridSearchCV(SVC(class_weight=weights), scoring=score, param_grid=tuned_parameters,
                           n_jobs=-1, cv=kf, refit=True, verbose=0)
        clf.fit(X, y)
        print(score, 'Best Params: {}'.format(clf.best_params_))
else:
    print('건너뜀 (RUN_GRID_SEARCH=False)')

## 7. sklearn 준지도 학습 ([코드 50])

가이드북 [코드 50]의 루프를 그대로 따릅니다.
1. labeled 학습 데이터로 모델 학습 → 평가 데이터로 평가
2. 남은 unlabeled 데이터를 예측하고 **확신도(가장 큰 클래스 확률)** 기준 내림차순 정렬
3. 상위 `PERCENTAGE`%에 예측값을 pseudo-label로 붙여 labeled 학습 데이터에 합침
4. 남은 unlabeled가 `(100 - UNLABELED_USAGE)`% 아래로 줄어들 때까지 반복

모델 설정
- SVM: 가이드북 [코드 50] 그대로 `SVC(C=0.001, kernel='rbf', gamma=1e-2, class_weight={0:100, 1:1}, probability=True)`
- RF: 가이드북 [코드 49] 탐색 결과 파라미터 + 같은 class_weight (가이드북에 RF 전체 코드가 없어 SVM과 동일하게 가정)
- GNB: 기본값

In [ ]:
weights = {0: 100.0, 1: 1.0}   # 가이드북 [코드 50] 그대로 (양품 쪽 가중치가 100)

MODEL_FACTORY = {
    'SVM': lambda: SVC(C=0.001, kernel='rbf', gamma=1e-2, class_weight=weights,
                       probability=True, random_state=SEED),
    'RF':  lambda: RandomForestClassifier(bootstrap=False, max_depth=10, max_features='sqrt',
                                          min_samples_leaf=1, min_samples_split=5, n_estimators=1200,
                                          class_weight=weights, n_jobs=-1, random_state=SEED),
    'GNB': lambda: GaussianNB(),
}

def train_and_evaluate(make_model, percentage=PERCENTAGE, unlabeled_usage=UNLABELED_USAGE, verbose=True):
    X_test, Y_test = data.moldset_labeled_test_X, data.moldset_labeled_test_Y
    X_with_label = data.moldset_labeled_train_X.copy()
    Y_with_label = data.moldset_labeled_train_Y.copy()
    without_label = data.moldset_unlabeled.copy()

    # 남은 클래스 변수가 존재하지 않는 데이터의 개수
    num_left_unlabeled = int(without_label.shape[0] * (100 - unlabeled_usage) * 0.01)
    history = []

    while True:
        model = make_model()
        model.fit(X_with_label, Y_with_label)
        y_pred = model.predict(X_test)
        y_score = model.predict_proba(X_test)[:, 1]
        history.append({'labeled_rows': len(X_with_label),
                        'pseudo_1': int(Y_with_label.sum() - data.moldset_labeled_train_Y.sum()),
                        **score_dict(Y_test, y_pred, y_score)})

        if without_label.shape[0] < num_left_unlabeled:
            break

        # Unlabeled 데이터에 대한 예측 → 확신도 내림차순 정렬
        prob = model.predict_proba(without_label)
        without_label = without_label.assign(confidence=confident_prediction(prob))
        without_label = without_label.sort_values(by=['confidence'], ascending=False)
        cutting_index = max(1, int(without_label.shape[0] * percentage * 0.01))
        chosen = without_label.iloc[:cutting_index].drop(columns=['confidence'])
        without_label = without_label.iloc[cutting_index:].drop(columns=['confidence'])
        pseudo_label = pd.Series(model.predict(chosen), name='PassOrFail')

        # Labeled 데이터와 통합 후 학습
        X_with_label = pd.concat([X_with_label, chosen], ignore_index=True)
        Y_with_label = pd.concat([Y_with_label, pseudo_label], ignore_index=True)

    if verbose:
        evaluation(Y_test, y_pred)
        print()
        print('ALL DONE, UNLABELED USED: {:.0%}'.format(1 - without_label.shape[0] / data.moldset_unlabeled.shape[0]))
    return model, pd.DataFrame(history)

results, histories = {}, {}
for name in RUN_SKLEARN:
    print(f'===== {name} =====')
    model, hist = train_and_evaluate(MODEL_FACTORY[name])
    histories[name] = hist
    results[name] = hist.iloc[-1].to_dict()
    print()

### 7-1. 반복별 성능 변화

pseudo-label이 쌓이면서 평가 성능이 좋아지는지(= 준지도 학습 효과가 있는지) 확인합니다.
첫 번째 점(반복 0)이 **labeled 데이터만으로 학습한 베이스라인**입니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, hist in histories.items():
    x = range(len(hist))
    axes[0].plot(x, hist['재현율'], marker='o', label=name)
    axes[1].plot(x, hist['정밀도'], marker='o', label=name)
    axes[2].plot(x, hist['PR-AUC(확률 기준)'], marker='o', label=name)
for ax, t in zip(axes, ['Recall', 'Precision', 'PR-AUC']):
    ax.set_title(t); ax.set_xlabel('iteration (0 = labeled only)'); ax.set_ylim(-0.05, 1.05); ax.legend()
plt.tight_layout(); plt.show()

for name, hist in histories.items():
    print(name, '| pseudo-label로 붙은 불량(1) 수:', int(hist['pseudo_1'].iloc[-1]),
          '/ 최종 학습 행 수:', int(hist['labeled_rows'].iloc[-1]))

## 8. Keras 심층 신경망 준지도 학습 ([코드 53], [코드 54])

구조: Dense(32) → Dense(64) → Dropout(0.25) → Dense(32) → Dropout(0.2) → Dense(16) → Dense(1, sigmoid)

반복마다 최대 `DNN_EPOCHS_PER_ITER`(가이드북: 100) epoch 학습하며, `val_precision`과 `val_recall`이 10 epoch 동안 함께 좋아지지 않으면 멈춥니다 (가이드북 로직 그대로).

⚠️ 가이드북 조건은 `>=` 비교라서, 검증 데이터에서 불량을 하나도 못 맞히면(precision·recall이 계속 0) 멈추지 않고 매번 끝까지 학습합니다. 오래 걸리면 `DNN_EPOCHS_PER_ITER`를 줄이세요.

In [ ]:
if RUN_DNN:
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # TensorFlow 로그 숨기기
    import tensorflow as tf
    from tensorflow.keras.models import Sequential, load_model
    from tensorflow.keras.layers import Dense, Dropout
    tf.random.set_seed(SEED)

    def build_model(input_dim):
        model = Sequential()
        model.add(tf.keras.Input(shape=(input_dim,)))
        model.add(Dense(units=32, activation='relu'))
        model.add(Dense(units=64, activation='relu'))
        model.add(Dropout(0.25))
        model.add(Dense(units=32, activation='relu'))
        model.add(Dropout(0.2))
        model.add(Dense(units=16, activation='relu'))
        model.add(Dense(units=1, activation='sigmoid'))
        model.compile(loss='binary_crossentropy', optimizer='adam',
                      metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
                               tf.keras.metrics.Recall(name='recall')])
        return model

    def train_and_evaluate_nn(percentage=PERCENTAGE, unlabeled_usage=UNLABELED_USAGE, patience=10):
        X_test, Y_test = data.moldset_labeled_test_X, data.moldset_labeled_test_Y
        X_with_label = data.moldset_labeled_train_X.copy()
        Y_with_label = data.moldset_labeled_train_Y.copy()
        without_label = data.moldset_unlabeled.copy()
        num_left_unlabeled = int(without_label.shape[0] * (100 - unlabeled_usage) * 0.01)

        model = build_model(len(FEATURES))
        history_rows = []
        while True:
            # 심층 신경망 모델을 이용하여 학습 진행
            val_precision, val_recall, cnt = 0, 0, 0
            for epoch in range(DNN_EPOCHS_PER_ITER):
                h = model.fit(X_with_label, Y_with_label, epochs=1, validation_split=0.3, verbose=0).history
                if cnt >= patience:
                    break
                if h['val_precision'][0] >= val_precision and h['val_recall'][0] >= val_recall:
                    val_precision, val_recall, cnt = h['val_precision'][0], h['val_recall'][0], 0
                else:
                    cnt += 1

            y_score = model.predict(X_test, verbose=0).ravel()
            y_pred = predict(y_score)
            history_rows.append({'labeled_rows': len(X_with_label),
                                 'pseudo_1': int(Y_with_label.sum() - data.moldset_labeled_train_Y.sum()),
                                 **score_dict(Y_test, y_pred, y_score)})

            if without_label.shape[0] < num_left_unlabeled:
                break

            # 목표값이 없는 데이터에 대한 예측 실행
            prob = model.predict(without_label, verbose=0)
            without_label = without_label.assign(confidence=confident_prediction_1(prob))
            without_label = without_label.sort_values(by=['confidence'], ascending=False)
            cutting_index = max(1, int(without_label.shape[0] * percentage * 0.01))
            chosen = without_label.iloc[:cutting_index].drop(columns=['confidence'])
            without_label = without_label.iloc[cutting_index:].drop(columns=['confidence'])
            pseudo_label = pd.Series(predict(model.predict(chosen, verbose=0)), name='PassOrFail')

            # UPDATE
            X_with_label = pd.concat([X_with_label, chosen], ignore_index=True)
            Y_with_label = pd.concat([Y_with_label, pseudo_label], ignore_index=True)

        print('ALL DONE, UNLABELED USED: {:.0%}'.format(1 - without_label.shape[0] / data.moldset_unlabeled.shape[0]))
        model.save(f'NN_{PRODUCT}.keras')
        return model, pd.DataFrame(history_rows)

    nn_model, nn_hist = train_and_evaluate_nn()
    histories['DNN'] = nn_hist
    results['DNN'] = nn_hist.iloc[-1].to_dict()

In [ ]:
# [코드 53-7], [코드 54] 저장한 모델 불러와서 평가
if RUN_DNN:
    model = load_model(f'NN_{PRODUCT}.keras')
    y_pred = predict(model.predict(data.moldset_labeled_test_X, verbose=0))
    evaluation(data.moldset_labeled_test_Y, y_pred)

## 9. 결과 비교 ([표 4] 형식)

- **베이스라인** = 반복 0 (labeled만 학습), **준지도** = 마지막 반복
- 불량을 놓치지 않는 것이 중요하므로 **재현율**과 **PR-AUC**를 우선해서 봅니다.
- 가이드북 [표 4] (RG3): SVM 정확도 0.85 / 재현율 0.33, RF 0.98 / 1.00, GNB 0.22 / 1.00, DNN 0.98 / 1.00

In [ ]:
cols = ['정확도', '정밀도', '재현율', 'ROC-AUC(0/1 기준)', 'F1', 'PR-AUC(확률 기준)', 'TN', 'FP', 'FN', 'TP']
rows = []
for name, hist in histories.items():
    rows.append({'모델': name, '단계': '베이스라인(labeled만)', **hist.iloc[0][cols].to_dict()})
    rows.append({'모델': name, '단계': '준지도(최종)', **hist.iloc[-1][cols].to_dict()})
summary = pd.DataFrame(rows).set_index(['모델', '단계'])
print(f'PRODUCT={PRODUCT} | SPLIT_MODE={SPLIT_MODE} | UNLABELED_SAMPLE={UNLABELED_SAMPLE}')
summary.round(3)

## 10. 해석할 때 주의할 점

1. **스케일 기준 차이**: 가이드북 [코드 43]은 파일마다 따로 StandardScaler를 적용했습니다. 같은 값이라도 labeled와 unlabeled에서 의미가 다를 수 있어, pseudo-label 품질에 영향을 줍니다.
2. **LH/RH 샷 구조**: 같은 샷의 두 행은 공정값이 완전히 같습니다. `SPLIT_MODE='guidebook'`에서는 한 샷이 train/test에 나뉘어 들어갈 수 있으니, `'shot'`과 결과를 비교해 보세요 (4장의 누수 행 수 참고).
3. **한쪽만 불량인 샷**: 공정값이 같은데 라벨이 다른 경우는 어떤 모델로도 구분할 수 없습니다 (특히 RG3).
4. **class_weight**: 가이드북 SVM 설정은 양품(0)에 가중치 100을 줍니다. 불량 재현율을 높이려면 반대 방향(`{0:1, 1:큰 값}`)도 실험해볼 만합니다.
5. **pseudo-label 편향**: 확신도 상위만 뽑으면 대부분 양품(0)이 붙습니다. 7-1의 "pseudo-label로 붙은 불량 수"를 확인하세요.